# 第7章（续）：自定义训练循环与高级用法

**学习路线：**

| 阶段 | 章节 | 内容 | 关键词 |
|------|------|------|--------|
| **一、自动微分** | §6 | 计算图、backward()、梯度访问 | requires_grad, backward, grad |
| **二、标准训练循环** | §7 | MNIST 完整训练示例 | train/eval, DataLoader, step |
| **三、性能优化** | §8-§9 | torch.compile、混合精度 | torch.compile, autocast, GradScaler |
| **四、封装训练器** | §10 | Trainer 类、回调支持 | Trainer, checkpoint, callback |

> **前置要求**：请先学完 `7.1-7.3 模型构建与内置训练.ipynb`。
>
> 本笔记本使用 MNIST 真实数据集，所有代码可直接运行。

In [ ]:
# === 基础导入 ===
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")

---
# 第一阶段：自动微分详解

对应 Keras 的 `tf.GradientTape`。

## §6 PyTorch 的自动微分机制

### Keras vs PyTorch 的梯度计算

```
Keras (tf.GradientTape):
  with tf.GradientTape() as tape:
      logits = model(x)              # 1. 记录前向传播
      loss = loss_fn(y, logits)       # 2. 计算损失
  grads = tape.gradient(loss, w)     # 3. 反向传播求梯度

PyTorch (autograd):
  outputs = model(x)                  # 1. 前向传播（自动记录计算图）
  loss = loss_fn(outputs, y)          # 2. 计算损失
  loss.backward()                     # 3. 反向传播（梯度存入 .grad）
  optimizer.step()                    # 4. 更新权重
```

核心区别：
- Keras 需要显式 `with tape:` 包裹
- PyTorch **自动追踪**所有带 `requires_grad=True` 的张量

### 6.1 `requires_grad` 机制

In [ ]:
# 6.1 requires_grad 控制梯度追踪

# 默认：创建时不追踪梯度
x = torch.tensor([1.0, 2.0, 3.0])
print(f"默认 requires_grad: {x.requires_grad}")  # False

# 开启梯度追踪
x_grad = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
print(f"requires_grad=True: {x_grad.requires_grad}")  # True

# 对已有张量开启追踪
x.requires_grad_(True)  # 原地修改
print(f"原地开启 requires_grad: {x.requires_grad}")  # True

### 6.2 计算图与反向传播

In [ ]:
# 6.2 计算图演示

x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

# 定义计算：z = x² + xy + y²
z = x**2 + x*y + y**2

print(f"z = {z.item()}")  # 4 + 6 + 9 = 19
print(f"z 的 grad_fn: {z.grad_fn}")  # 记录计算历史

# 反向传播
z.backward()

# 查看梯度
# dz/dx = 2x + y = 2*2 + 3 = 7
# dz/dy = x + 2y = 2 + 2*3 = 8
print(f"dz/dx = {x.grad.item()}")  # 7.0
print(f"dz/dy = {y.grad.item()}")  # 8.0

### 6.3 访问模型的梯度

In [ ]:
# 6.3 访问模型的梯度
model = nn.Linear(10, 5)

# 模拟一次前向+反向
x = torch.randn(3, 10)
y = torch.randint(0, 5, (3,))
output = model(x)
loss = nn.CrossEntropyLoss()(output, y)
loss.backward()

# 查看梯度
print("权重梯度：")
for name, param in model.named_parameters():
    if param.grad is not None:
        print(f"  {name}: grad shape={tuple(param.grad.shape)}, grad norm={param.grad.norm().item():.4f}")

### 6.4 梯度裁剪（防止梯度爆炸）

In [ ]:
# 6.4 梯度裁剪

# 方式一：按范数裁剪（最常用）
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
print(f"裁剪后梯度范数: {sum(p.grad.norm().item()**2 for p in model.parameters() if p.grad is not None)**0.5:.4f}")

# 方式二：按值裁剪
# torch.nn.utils.clip_grad_value_(model.parameters(), clip_value=0.5)

print("\n梯度裁剪在处理 RNN/LSTM/深层网络时特别有用，防止梯度爆炸。")

---
# 第二阶段：标准训练循环详解

对应 Keras 的 `tf.GradientTape` 手写训练循环。

## §7 标准训练循环详解

### 何时需要修改训练循环？

| 场景 | PyTorch 需要改？ | 说明 |
|------|-----------------|------|
| 标准分类/回归 | ❌ 不需要 | 标准循环已够用 |
| 多任务加权损失 | ✅ 需要 | 自定义 loss 组合 |
| GAN 交替训练 | ✅ 需要 | 多个优化器交替 |
| 梯度裁剪/惩罚 | ✅ 需要 | 在 `backward()` 后加入 |
| 元学习/双层优化 | ✅ 需要 | 复杂梯度计算 |

**PyTorch 的核心优势**：训练循环本身就是你的代码，想改哪里改哪里。
不像 Keras 需要打破 `fit()` 的抽象层。

### 7.1 准备数据和模型

In [ ]:
# 7.1 加载 MNIST 数据
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform)

train_size = 50000
val_size = 10000
train_subset, val_subset = torch.utils.data.random_split(train_dataset, [train_size, val_size])

batch_size = 64
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"训练集: {len(train_subset)} 样本")
print(f"验证集: {len(val_subset)} 样本")
print(f"测试集:  {len(test_dataset)} 样本")

In [ ]:
# 7.2 构建模型和优化器
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 10)
).to(device)

optimizer = optim.RMSprop(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

print(f"模型已创建，使用设备: {device}")

### 7.3 完整自定义训练循环

```
for epoch in range(epochs):
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()          # 1. 清零梯度
        outputs = model(batch_x)       # 2. 前向传播
        loss = loss_fn(outputs, y)     # 3. 计算损失
        loss.backward()                # 4. 反向传播（计算梯度）
        optimizer.step()               # 5. 更新权重

    for val_x, val_y in val_loader:    # 6. 验证阶段
        val_outputs = model(val_x)
        ...
    
    打印指标 → 重置指标
```

In [ ]:
# 7.3 完全自定义训练循环
epochs = 3

for epoch in range(epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch + 1}/{epochs}")
    print(f"{'='*50}")
    
    # ---- 训练阶段 ----
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for step, (batch_x, batch_y) in enumerate(train_loader):
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        # 1. 清零梯度
        optimizer.zero_grad()
        
        # 2. 前向传播（training=True 的等价操作：model.train() 已设置）
        outputs = model(batch_x)
        
        # 3. 计算损失
        loss = loss_fn(outputs, batch_y)
        
        # 4. 反向传播（等价于 tape.gradient）
        loss.backward()
        
        # 5. 更新权重（等价于 optimizer.apply_gradients）
        optimizer.step()
        
        # 统计
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)
        
        if step % 100 == 0:
            print(f"  Step {step:4d}: loss = {loss.item():.4f}")
    
    train_acc = correct / total
    avg_loss = total_loss / len(train_loader)
    print(f"\n  训练损失: {avg_loss:.4f}, 训练精度: {train_acc:.4f}")
    
    # ---- 验证阶段 ----
    model.eval()
    val_correct = 0
    val_total = 0
    val_loss = 0
    
    with torch.no_grad():  # 不计算梯度，节省内存
        for val_x, val_y in val_loader:
            val_x = val_x.to(device)
            val_y = val_y.to(device)
            
            val_outputs = model(val_x)
            val_loss += loss_fn(val_outputs, val_y).item()
            _, predicted = torch.max(val_outputs, 1)
            val_correct += (predicted == val_y).sum().item()
            val_total += val_y.size(0)
    
    val_acc = val_correct / val_total
    val_avg_loss = val_loss / len(val_loader)
    print(f"  验证损失: {val_avg_loss:.4f}, 验证精度: {val_acc:.4f}")

### 关键概念：PyTorch 自动微分 vs tf.GradientTape

```
tf.GradientTape（显式记录）：
  普通代码：  y = f(x)     → 只计算结果
  with tape:  y = f(x)     → 记录计算过程
              grads = tape.gradient(y, x)  → 反向传播

PyTorch autograd（自动记录）：
  普通代码：  y = f(x)     → 自动记录（如果 x.requires_grad=True）
              y.backward() → 梯度存入各张量的 .grad 属性

类比：
  Keras: 手动打开录音机 → 说话 → 停止录音 → 播放
  PyTorch: 录音机一直开着，需要时才保存
```

---
# 第三阶段：性能优化

对应 Keras 的 `tf.function` 加速。

## §8 利用 `torch.compile()` 加快运行速度

PyTorch 2.0+ 引入了 `torch.compile()`，将模型编译为优化后的计算图。

```
普通 Python 代码：逐行解释执行（慢）
     ↓ torch.compile()
编译为优化图：TorchDynamo 追踪操作，TorchInductor 编译
     ↓
全局优化：算子融合、内存优化、并行计算
     ↓
直接执行编译好的图（快）
```

**注意**：第一次调用时需要编译（有开销），后续调用直接使用编译好的图。

In [ ]:
# 8.1 使用 torch.compile() 编译模型

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 10)
).to(device)

if hasattr(torch, 'compile'):
    compiled_model = torch.compile(model)
    print("模型已用 torch.compile() 编译")
    print("第一次调用会稍慢（编译开销），后续调用更快。")
else:
    compiled_model = model
    print("当前 PyTorch 版本不支持 torch.compile()（需要 2.0+）")
    print("使用原始模型，不影响功能。")

optimizer = optim.RMSprop(compiled_model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
# 8.2 用编译后的模型进行训练
epochs = 3

for epoch in range(epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch + 1}/{epochs} {'(使用 torch.compile 加速)' if hasattr(torch, 'compile') else ''}")
    print(f"{'='*50}")
    
    compiled_model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for step, (batch_x, batch_y) in enumerate(train_loader):
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = compiled_model(batch_x)
        loss = loss_fn(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)
        
        if step % 100 == 0:
            print(f"  Step {step:4d}: loss = {loss.item():.4f}")
    
    train_acc = correct / total
    print(f"\n  训练精度: {train_acc:.4f}")
    
    # 验证
    compiled_model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for val_x, val_y in val_loader:
            val_x = val_x.to(device)
            val_y = val_y.to(device)
            val_outputs = compiled_model(val_x)
            _, predicted = torch.max(val_outputs, 1)
            val_correct += (predicted == val_y).sum().item()
            val_total += val_y.size(0)
    
    val_acc = val_correct / val_total
    print(f"  验证精度: {val_acc:.4f}")

## §9 混合精度训练

使用 `torch.cuda.amp` 可以显著加速 GPU 训练并减少显存占用。

```
标准精度： 全部使用 float32（安全但慢）
混合精度： 前向用 float16（快），反向部分用 float32（安全）
```

两个核心组件：
- `autocast`：自动选择精度
- `GradScaler`：防止小梯度在 float16 下溢出

In [ ]:
# 9.1 混合精度训练示例（需要 GPU）

if torch.cuda.is_available():
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, 10)
    ).to(device)
    
    optimizer = optim.RMSprop(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    
    # 创建 GradScaler
    scaler = torch.amp.GradScaler('cuda')
    
    epochs = 2
    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs} (混合精度训练)")
        model.train()
        total_loss = 0
        
        for step, (batch_x, batch_y) in enumerate(train_loader):
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            
            optimizer.zero_grad()
            
            # autocast：前向传播自动选择精度
            with torch.amp.autocast('cuda'):
                outputs = model(batch_x)
                loss = loss_fn(outputs, batch_y)
            
            # GradScaler：缩放 loss 并反向传播
            scaler.scale(loss).backward()
            
            # 更新权重（自动 unscale）
            scaler.step(optimizer)
            scaler.update()
            
            total_loss += loss.item()
            if step % 200 == 0:
                print(f"  Step {step:4d}: loss = {loss.item():.4f}")
        
        print(f"  平均损失: {total_loss/len(train_loader):.4f}")
    
    print("\n混合精度训练完成！")
else:
    print("混合精度训练需要 GPU，跳过此节。")
    print("在有 GPU 的环境中，使用 torch.amp.autocast + GradScaler 即可。")

### 关键区别：`torch.no_grad()` vs `model.eval()`

| 操作 | 作用 | 何时使用 |
|------|------|----------|
| `model.eval()` | 设置层行为（关闭 Dropout, BN 使用全局统计） | 验证/推理前调用 |
| `torch.no_grad()` | 不计算梯度，节省内存和时间 | 验证/推理时用 `with` 包裹 |
| `model.train()` | 恢复训练行为（开启 Dropout, BN 更新统计） | 训练前调用 |

**两者必须同时使用**：`model.eval()` 不阻止梯度计算，`no_grad()` 不改变层行为。

---
# 第四阶段：封装训练循环

对应 Keras 的覆盖 `train_step()` —— 但 PyTorch 中我们封装成 Trainer 类。

## §10 Trainer 类封装

将训练循环封装为一个可复用的 `Trainer` 类：
- 支持回调
- 支持验证和测试
- 支持检查点保存
- 保持代码整洁

In [ ]:
# 10.1 回调基类（复用之前的定义）

class Callback:
    """回调函数基类"""
    def on_train_begin(self, trainer): pass
    def on_train_end(self, trainer): pass
    def on_epoch_begin(self, epoch, trainer): pass
    def on_epoch_end(self, epoch, trainer): pass


class EarlyStopping(Callback):
    """早停回调"""
    def __init__(self, monitor="val_loss", patience=2, mode="min"):
        self.monitor = monitor
        self.patience = patience
        self.mode = mode
        self.counter = 0
        self.best_value = None
    
    def on_epoch_end(self, epoch, trainer):
        current = trainer.metrics.get(self.monitor)
        if current is None:
            return
        
        if self.best_value is None:
            self.best_value = current
        elif (self.mode == "min" and current < self.best_value) or \
             (self.mode == "max" and current > self.best_value):
            self.best_value = current
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                trainer.should_stop = True
                print(f"\n  [EarlyStopping] {self.monitor} 在 {self.patience} 轮内未改善")


class ModelCheckpoint(Callback):
    """模型检查点"""
    def __init__(self, filepath, monitor="val_loss", mode="min"):
        self.filepath = filepath
        self.monitor = monitor
        self.mode = mode
        self.best_value = None
    
    def on_epoch_end(self, epoch, trainer):
        current = trainer.metrics.get(self.monitor)
        if current is None:
            return
        
        if self.best_value is None or \
           (self.mode == "min" and current < self.best_value) or \
           (self.mode == "max" and current > self.best_value):
            self.best_value = current
            torch.save({
                'epoch': epoch,
                'model_state_dict': trainer.model.state_dict(),
                'optimizer_state_dict': trainer.optimizer.state_dict(),
                self.monitor: current,
            }, self.filepath)
            print(f"  [Checkpoint] 保存到 {self.filepath}")


print("回调类定义完成")

In [ ]:
# 10.2 Trainer 类

class Trainer:
    """
    PyTorch 训练器。
    
    封装了训练循环，同时保持灵活性。
    类似 Keras 的 fit()，但更透明、更容易扩展。
    """
    
    def __init__(self, model, optimizer, loss_fn, device, callbacks=None):
        self.model = model
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.device = device
        self.callbacks = callbacks or []
        self.metrics = {}
        self.should_stop = False
        self.history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    
    def train_epoch(self, train_loader):
        """训练一个 epoch"""
        self.model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(self.device)
            batch_y = batch_y.to(self.device)
            
            self.optimizer.zero_grad()
            outputs = self.model(batch_x)
            loss = self.loss_fn(outputs, batch_y)
            loss.backward()
            self.optimizer.step()
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)
        
        return total_loss / len(train_loader), correct / total
    
    @torch.no_grad()
    def validate(self, val_loader):
        """验证模型"""
        self.model.eval()
        total_loss = 0
        correct = 0
        total = 0
        
        for batch_x, batch_y in val_loader:
            batch_x = batch_x.to(self.device)
            batch_y = batch_y.to(self.device)
            
            outputs = self.model(batch_x)
            loss = self.loss_fn(outputs, batch_y)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)
        
        return total_loss / len(val_loader), correct / total
    
    def fit(self, train_loader, val_loader, epochs):
        """
        训练模型。
        
        参数
        ----
        train_loader : DataLoader
            训练数据加载器
        val_loader : DataLoader
            验证数据加载器
        epochs : int
            训练轮数
        """
        self.model = self.model.to(self.device)
        
        # on_train_begin
        for cb in self.callbacks:
            cb.on_train_begin(self)
        
        for epoch in range(epochs):
            # on_epoch_begin
            for cb in self.callbacks:
                cb.on_epoch_begin(epoch, self)
            
            # 训练 + 验证
            train_loss, train_acc = self.train_epoch(train_loader)
            val_loss, val_acc = self.validate(val_loader)
            
            self.metrics = {
                "train_loss": train_loss, "train_acc": train_acc,
                "val_loss": val_loss, "val_acc": val_acc
            }
            
            print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, "
                  f"val_acc={val_acc:.4f}")
            
            self.history["train_loss"].append(train_loss)
            self.history["train_acc"].append(train_acc)
            self.history["val_loss"].append(val_loss)
            self.history["val_acc"].append(val_acc)
            
            # on_epoch_end
            for cb in self.callbacks:
                cb.on_epoch_end(epoch, self)
            
            if self.should_stop:
                print(f"训练在第 {epoch+1} 轮提前结束")
                break
        
        # on_train_end
        for cb in self.callbacks:
            cb.on_train_end(self)
    
    @torch.no_grad()
    def evaluate(self, test_loader):
        """对应 Keras 的 evaluate()"""
        self.model.eval()
        loss, acc = self.validate(test_loader)
        print(f"测试集 - loss: {loss:.4f}, accuracy: {acc:.4f}")
        return loss, acc
    
    @torch.no_grad()
    def predict(self, dataloader, num_samples=5):
        """对应 Keras 的 predict()"""
        self.model.eval()
        images, labels = next(iter(dataloader))
        images = images[:num_samples].to(self.device)
        labels = labels[:num_samples]
        
        outputs = self.model(images)
        probs = torch.softmax(outputs, dim=1)
        pred_classes = torch.argmax(probs, dim=1).cpu()
        
        print(f"{'样本':>6} | {'预测':>6} | {'概率':>8} | {'真实':>6}")
        print("-" * 42)
        for i in range(num_samples):
            match = "✓" if pred_classes[i] == labels[i] else "✗"
            print(f"  #{i:>4} | {pred_classes[i]:>6} | {probs[i, pred_classes[i]].item():>8.4f} | {labels[i]:>6} {match}")


print("Trainer 类定义完成")

In [ ]:
# 10.3 使用 Trainer 训练

# 重新创建模型
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 10)
)

optimizer = optim.RMSprop(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

# 创建 Trainer
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device=device,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=2, mode="min"),
        ModelCheckpoint(filepath="trainer_best.pt", monitor="val_loss", mode="min")
    ]
)

# 训练（类似 Keras 的 fit()）
print("使用 Trainer 训练（最多 8 轮）：")
trainer.fit(train_loader, val_loader, epochs=8)

In [ ]:
# 10.4 评估和预测
print("=== 测试集评估 ===")
trainer.evaluate(test_loader)

print("\n=== 预测样本 ===")
trainer.predict(test_loader)

## §11 PyTorch Lightning 简介

如果你想要类似 Keras `fit()` 的体验，但又不想自己写 Trainer，
那么 **PyTorch Lightning** 是最佳选择。

```
纯 PyTorch:  手写训练循环 + 验证循环 + 回调 + checkpoint
     ↓
Lightning:   LightningModule + Trainer 自动完成一切
     ↓
效果:        类似 Keras 的 fit()，但保留 PyTorch 的灵活性
```

In [ ]:
# 11.1 Lightning 示例代码（需要安装 pytorch-lightning）

lightning_code = '''
# pip install pytorch-lightning
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

class MNISTModel(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 10)
        )
    
    def forward(self, x):
        return self.model(x)
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = nn.CrossEntropyLoss()(logits, y)
        self.log("train_loss", loss)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = nn.CrossEntropyLoss()(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("val_loss", loss)
        self.log("val_acc", acc)
    
    def configure_optimizers(self):
        return optim.RMSprop(self.parameters(), lr=1e-3)

# 使用方式：
# model = MNISTModel()
# trainer = pl.Trainer(
#     max_epochs=10,
#     callbacks=[
#         EarlyStopping(monitor="val_loss", patience=2),
#         ModelCheckpoint(monitor="val_loss")
#     ]
# )
# trainer.fit(model, train_loader, val_loader)
'''

print("PyTorch Lightning 代码示例：")
print(lightning_code)

---
# 总结

### PyTorch 训练方式对比与选择

| 方式 | 代码量 | 灵活性 | 适用场景 |
|------|--------|--------|----------|
| **手写训练循环** | 中等 | 最高 | 所有场景，推荐学习 |
| **Trainer 封装** | 最少 | 高 | 日常训练，类似 fit() |
| **PyTorch Lightning** | 少 | 高 | 大型项目、分布式训练 |

### 选择建议

```
你想做什么？
│
├─ 学习 PyTorch 原理 → 手写训练循环，理解每一步
│
├─ 日常快速训练 → Trainer 封装或 Lightning
│
├─ 需要完全控制（GAN、元学习）？
│   └─→ 手写循环，修改任意步骤
│
└─ 大型项目/分布式？
    └─→ PyTorch Lightning
```

### Keras vs PyTorch 训练方式对比

| 组件 | Keras | PyTorch |
|------|-------|---------|
| 自动微分 | `tf.GradientTape` | `loss.backward()` |
| 梯度应用 | `optimizer.apply_gradients()` | `optimizer.step()` |
| 编译加速 | `@tf.function` | `torch.compile()` |
| 训练封装 | 覆盖 `train_step()` | Trainer 类 / Lightning |
| 检查点 | `model.save()` / `ModelCheckpoint` | `torch.save()` |
| 多 GPU | `MirroredStrategy` | `DataParallel` / `DDP` |

### 第7章 总回顾

**模型构建（7.1-7.2 节）**
```
简单模型     → nn.Sequential
复杂模型     → nn.Module 子类化（推荐首选）
动态层       → ModuleList / ModuleDict
```

**训练流程（7.3 节）**
```
optimizer + loss_fn + 训练循环 + 验证循环  是标准流程
model.train() / model.eval()  控制层行为
torch.no_grad()               跳过梯度计算
```

**训练扩展（7.4 节）**
```
性能优化 → torch.compile() + 混合精度
封装训练 → Trainer 类 + 回调系统
进阶框架 → PyTorch Lightning
```

---
**恭喜完成第7章！PyTorch 的核心思想已经掌握，接下来可以继续学习更高级的深度学习主题。**